# 01 — WordNet Filtering and Split Assignment

**Primary author:** Victoria

**Builds on:**
- *01_data_cleaning.ipynb* (Victoria — WordNet lookup logic, article stripping, underscore conversion)
- *structural_filtering.ipynb* (Victoria — produces `clues_filtered.csv`, the input to this notebook)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

---

This notebook transforms `clues_filtered.csv` into the foundational dataset
for the custom embedding model pipeline. Every downstream step — phrase
construction, triplet building, embedding generation, and ATE evaluation —
requires that both the definition and answer of a clue have at least one
WordNet synset, because our phrase construction strategies (f functions) build
sense-specific text passages from WordNet resources (definitions and usage
examples). Rows without synsets cannot participate in any of those strategies,
so we filter them here.

The notebook also assigns the 30/20/50 train/validate/test split at the
(definition, answer) pair level, ensuring no leakage: if a particular
(definition, answer) pair appears in multiple clues, all those clues land in
the same split. Finally, it constructs the canonical vocabulary files whose
row ordering serves as the index for all downstream embedding arrays.

**Reads:** `../../data/clues_filtered.csv` (457,262 rows)
**Writes to:** `data/filtered_split/wn_synset/`
- `clues_wn_filtered.csv` — filtered rows with `definition_wn`, `answer_wn`, and `split` columns
- `clues_val.csv` — convenience subset (validation split only)
- `vocabulary.csv` — all unique words; canonical ordering = embedding index
- `vocabulary_val.csv` — validation-split subset; own 0-indexed ordering

---

## §1 — Imports and Configuration

Environment auto-detection lets this notebook run unmodified on Local, Great
Lakes, and Colab. Path variables are defined once here so that every
subsequent cell references the same roots.

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================
import time
import pandas as pd
import numpy as np
from pathlib import Path

import nltk
from nltk.corpus import wordnet as wn
from sklearn.model_selection import train_test_split

try:
    wn.synsets("test")
except LookupError:
    nltk.download("wordnet", quiet=True)

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    PROJECT_ROOT = Path("../..").resolve()

COMPONENT_ROOT = PROJECT_ROOT / "custom_embedding_model"
DATA_DIR       = COMPONENT_ROOT / "data" / "filtered_split" / "wn_synset"
OUTPUT_DIR     = COMPONENT_ROOT / "outputs"

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment: {env_label}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR:     {DATA_DIR}")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# Version reporting (Decision 18)
# ============================================================
import sklearn

print(f"pandas:          {pd.__version__}")
print(f"numpy:           {np.__version__}")
print(f"scikit-learn:    {sklearn.__version__}")
print(f"nltk:            {nltk.__version__}")
print(f"WordNet corpus:  {wn.get_version()}")

In [ ]:
# ============================================================
# Load clues_filtered.csv
# ============================================================
t0 = time.time()

df = pd.read_csv(
    PROJECT_ROOT / "data" / "clues_filtered.csv",
    keep_default_na=False,
    na_values=[""],
    usecols=["row_id", "clue_id", "surface", "definition", "answer"],
)

assert len(df) == 457_262, f"Expected 457,262 rows, got {len(df):,}"
print(f"Loaded clues_filtered.csv: {len(df):,} rows")
df.head(3)

---

## §2 — WordNet Lookup Function

Many crossword definitions include a leading article ("a shade", "an animal",
"the law") or infinitive marker ("to flee") that is not part of the WordNet
headword. The lookup function tries the raw text first (lowercased, with
spaces converted to underscores), then strips the first matching prefix and
retries. This article-stripping step is critical for coverage: without it,
thousands of valid definitions would fail to resolve (quantified in §4).

The function returns three values — the successful lookup form, the synset
list, and which prefix was stripped — so we can audit recovery rates
downstream.

In [ ]:
# ============================================================
# WordNet lookup function
# ============================================================
STRIP_PREFIXES = [("a ", "a"), ("an ", "an"), ("the ", "the"), ("to ", "to")]


def wordnet_lookup(text):
    """Look up WordNet synsets for a definition or answer string.

    Returns (wn_form, synsets, strip_applied):
      - wn_form: the string form that succeeded, or None
      - synsets: list of synsets (may be empty)
      - strip_applied: which prefix was stripped (None, "a", "an", "the", "to")
    """
    lowered = str(text).lower()
    lookup = lowered.replace(" ", "_")
    synsets = wn.synsets(lookup)
    if synsets:
        return (lookup, synsets, None)

    for prefix, label in STRIP_PREFIXES:
        if lowered.startswith(prefix):
            remainder = lowered[len(prefix):].replace(" ", "_")
            synsets = wn.synsets(remainder)
            if synsets:
                return (remainder, synsets, label)
            break  # only try the first matching prefix

    return (None, [], None)


# --- Sanity checks ---
assert wordnet_lookup("shade")[0] == "shade" and wordnet_lookup("shade")[2] is None
assert wordnet_lookup("a shade")[2] == "a"
assert wordnet_lookup("an animal")[2] == "an"
assert wordnet_lookup("the law")[2] == "the"
assert wordnet_lookup("to flee")[2] == "to"
assert wordnet_lookup("PLOT")[0] == "plot"
assert wordnet_lookup("ice cream")[0] == "ice_cream"
assert wordnet_lookup("xyzzy")[0] is None

print("All sanity checks passed.")

---

## §3 — Apply WordNet Lookup to Definitions and Answers

We apply the lookup to unique definition and answer strings first, then map
results back to the full dataframe. This avoids redundant WordNet queries —
457K rows share far fewer unique strings — and keeps the lookup step fast.

In [ ]:
# ============================================================
# Apply WordNet lookup to unique definitions and answers
# ============================================================
t_wn = time.time()

# --- Definitions ---
unique_defs = df["definition"].unique()
print(f"Unique definitions: {len(unique_defs):,}")

def_results = {d: wordnet_lookup(d) for d in unique_defs}
df["definition_wn"] = df["definition"].map(lambda d: def_results[d][0])
df["def_strip"]     = df["definition"].map(lambda d: def_results[d][2])

# --- Answers ---
unique_ans = df["answer"].unique()
print(f"Unique answers:     {len(unique_ans):,}")

ans_results = {a: wordnet_lookup(a) for a in unique_ans}
df["answer_wn"] = df["answer"].map(lambda a: ans_results[a][0])
df["ans_strip"] = df["answer"].map(lambda a: ans_results[a][2])

elapsed_wn = time.time() - t_wn
print(f"\nWordNet lookups completed in {elapsed_wn:.1f}s")
print(f"  Definitions with synsets: {df['definition_wn'].notna().sum():,} / {len(df):,}")
print(f"  Answers with synsets:     {df['answer_wn'].notna().sum():,} / {len(df):,}")

# --- Show concrete examples ---
print("\nExample WordNet lookups (definitions):")
example_defs = ["shade", "a shade", "the law", "ice cream", "plot"]
for d in example_defs:
    wn_form, syns, strip = wordnet_lookup(d)
    syn_names = [s.name() for s in syns[:2]]
    print(f'  "{d}" -> wn_form="{wn_form}", strip={strip}, synsets={syn_names}...')


---

## §4 — Article-Stripping Diagnostic

Before filtering, we quantify how many unique definitions and answers were
recovered by each prefix strip. The Milestone II notebook only stripped
`"a "`; this pipeline expands to `"a "`, `"an "`, `"the "`, `"to "` (Decision
16). The diagnostic below shows whether the three new prefixes contribute
meaningful recovery or are negligible.

In [ ]:
# ============================================================
# Article-stripping diagnostic
# ============================================================
def strip_diagnostic(results_dict, label):
    """Print a breakdown of how unique values were resolved."""
    total = len(results_dict)
    no_strip = sum(1 for _, (wn_form, _, strip) in results_dict.items()
                   if wn_form is not None and strip is None)
    by_a   = sum(1 for _, (_, _, s) in results_dict.items() if s == "a")
    by_an  = sum(1 for _, (_, _, s) in results_dict.items() if s == "an")
    by_the = sum(1 for _, (_, _, s) in results_dict.items() if s == "the")
    by_to  = sum(1 for _, (_, _, s) in results_dict.items() if s == "to")
    failed = sum(1 for _, (wn_form, _, _) in results_dict.items() if wn_form is None)

    print(f"Article-stripping recovery (unique {label}):")
    print(f'  No strip needed:    {no_strip:>6,} ({no_strip/total:.1%})')
    print(f'  Recovered by "a":   {by_a:>6,} ({by_a/total:.1%})')
    print(f'  Recovered by "an":  {by_an:>6,} ({by_an/total:.1%})')
    print(f'  Recovered by "the": {by_the:>6,} ({by_the/total:.1%})')
    print(f'  Recovered by "to":  {by_to:>6,} ({by_to/total:.1%})')
    print(f'  No synsets found:   {failed:>6,} ({failed/total:.1%})')
    print()
    return {
        "no_strip": no_strip, "a": by_a, "an": by_an,
        "the": by_the, "to": by_to, "failed": failed, "total": total,
    }


def_diag = strip_diagnostic(def_results, "definitions")
ans_diag = strip_diagnostic(ans_results, "answers")

---

## §5 — Filter to Rows with WordNet Coverage

We keep only rows where both `definition_wn` and `answer_wn` resolved
successfully. We expect to lose roughly half the dataset here — many crossword
answers are proper nouns, abbreviations, or slang that WordNet does not cover.
The breakdown below separates coverage loss into definition-side, answer-side,
and both-side failures to identify where the bottleneck lies.

In [ ]:
# ============================================================
# Filter to rows with WordNet coverage
# ============================================================
n_before = len(df)
def_failed = df["definition_wn"].isna()
ans_failed = df["answer_wn"].isna()

n_def_only = int((def_failed & ~ans_failed).sum())
n_ans_only = int((~def_failed & ans_failed).sum())
n_both     = int((def_failed & ans_failed).sum())
n_any      = int((def_failed | ans_failed).sum())

print(f"Rows before filter:              {n_before:,}")
print(f"Rows dropped (definition failed): {n_def_only + n_both:,}  (def only: {n_def_only:,})")
print(f"Rows dropped (answer failed):     {n_ans_only + n_both:,}  (ans only: {n_ans_only:,})")
print(f"Rows dropped (both failed):       {n_both:,}")
print(f"Total rows dropped:               {n_any:,}")

df = df[~def_failed & ~ans_failed].copy()
df.drop(columns=["def_strip", "ans_strip"], inplace=True)

n_after = len(df)
print(f"Rows remaining after filter:     {n_after:,} ({n_after/n_before:.1%} of input)")

print(f"\n(52% retention is expected: many crossword answers are proper nouns,")
print(f" abbreviations, or informal terms absent from WordNet.)")


---

## §6 — Split Assignment

Splitting at the (definition, answer) pair level prevents data leakage: if
the same word pair appears in multiple clues (different surfaces, different
puzzles), all those rows land in the same split. Without this, the model
could memorize specific word associations during training and appear to
generalize on validation data that contains the same pairs.

The 30/20/50 ratio allocates the largest share to test (locked until a final
model is chosen), a moderate share to training, and the smallest share to
validation. Two calls to `train_test_split` with `random_state=42` produce
this three-way split deterministically.

In [ ]:
# ============================================================
# Split assignment at the (definition, answer) pair level
# ============================================================
pairs = df[["definition", "answer"]].drop_duplicates().reset_index(drop=True)
print(f"Unique (definition, answer) pairs: {len(pairs):,}")

remainder, test = train_test_split(pairs, test_size=0.50, random_state=42)
train, validate = train_test_split(remainder, test_size=0.40, random_state=42)

train["split"]    = "train"
validate["split"] = "validate"
test["split"]     = "test"

split_map = pd.concat([train, validate, test], ignore_index=True)
df = df.merge(split_map, on=["definition", "answer"], how="left")

# --- Validation ---
assert df["split"].notna().all(), "Some rows have no split assignment"

pair_splits = df.groupby(["definition", "answer"])["split"].nunique()
assert (pair_splits == 1).all(), "Some (definition, answer) pairs span multiple splits"

print("\nSplit distribution (pairs):")
for s in ["train", "validate", "test"]:
    n = (split_map["split"] == s).sum()
    print(f"  {s:>8}: {n:>7,} ({n/len(split_map):.1%})")

print("\nSplit distribution (rows):")
for s in ["train", "validate", "test"]:
    n = (df["split"] == s).sum()
    print(f"  {s:>8}: {n:>7,} ({n/len(df):.1%})")

# --- Show a few example rows with split labels ---
print("\nExample rows with split assignments:")
for _, row in df.sample(5, random_state=42).iterrows():
    print(f'  definition="{row["definition"]}", answer="{row["answer"]}", split={row["split"]}')


---

## §7 — Save `clues_wn_filtered.csv`

The primary output of this notebook. Every downstream notebook in the pipeline
reads from this file (or from subsets derived from it). The column ordering is
fixed and documented in DATA.md.

In [ ]:
# ============================================================
# Save clues_wn_filtered.csv
# ============================================================
output_cols = ["row_id", "clue_id", "surface", "definition", "answer",
               "definition_wn", "answer_wn", "split"]
df_out = df[output_cols]

assert list(df_out.columns) == output_cols
assert df_out["definition_wn"].notna().all()
assert df_out["answer_wn"].notna().all()
assert df_out["split"].notna().all()

out_path = DATA_DIR / "clues_wn_filtered.csv"
df_out.to_csv(out_path, index=False)
print(f"Saved {len(df_out):,} rows to {out_path.relative_to(COMPONENT_ROOT)}")

---

## §8 — Save `clues_val.csv`

A convenience subset containing only validation-split rows. Downstream
evaluation notebooks load this directly rather than filtering
`clues_wn_filtered.csv` each time — a small optimization that also reduces
the risk of accidentally including train or test rows in validation analysis.

In [ ]:
# ============================================================
# Save clues_val.csv
# ============================================================
df_val = df_out[df_out["split"] == "validate"]
val_path = DATA_DIR / "clues_val.csv"
df_val.to_csv(val_path, index=False)
print(f"Saved {len(df_val):,} validation rows to {val_path.relative_to(COMPONENT_ROOT)}")

---

## §9 — Vocabulary Construction

The unified vocabulary is the set of all unique words appearing as either
`definition_wn` or `answer_wn` in the filtered dataset. Definitions and
answers share a single vocabulary (Decision 4) because the same word can
appear as a definition in one clue and an answer in another.

The alphabetical sort order establishes the canonical row ordering that every
downstream embedding array uses as its index: row *k* in `vocabulary.csv`
corresponds to row *k* in any `.npy` embedding file indexed by this vocabulary.
This ordering is sacred and must never be changed once established.

In [ ]:
# ============================================================
# Build unified vocabulary
# ============================================================
all_def_words = set(df_out["definition_wn"].unique())
all_ans_words = set(df_out["answer_wn"].unique())
all_words = sorted(all_def_words | all_ans_words)

vocab = pd.DataFrame({"word": all_words, "row": range(len(all_words))})

assert vocab["word"].nunique() == len(vocab), "Duplicate words in vocabulary"
assert list(vocab["row"]) == list(range(len(vocab))), "Row values not contiguous 0..N-1"
assert all_def_words.issubset(set(vocab["word"])), "Missing definition words"
assert all_ans_words.issubset(set(vocab["word"])), "Missing answer words"

vocab_path = DATA_DIR / "vocabulary.csv"
vocab.to_csv(vocab_path, index=False)

only_def = all_def_words - all_ans_words
only_ans = all_ans_words - all_def_words
both = all_def_words & all_ans_words

print(f"Vocabulary size: {len(vocab):,}")
print(f"  Words as both def and ans: {len(both):,}")
print(f"  Words only as definition:  {len(only_def):,}")
print(f"  Words only as answer:      {len(only_ans):,}")
print(f"\nSaved to {vocab_path.relative_to(COMPONENT_ROOT)}")

# --- Show a few vocabulary entries ---
print("\nExample vocabulary entries:")
for _, row in vocab.sample(5, random_state=42).iterrows():
    role = "both" if row["word"] in both else ("def-only" if row["word"] in only_def else "ans-only")
    print(f'  row={row["row"]:>5}, word="{row["word"]}", role={role}')


---

## §10 — Validation Vocabulary Construction

The validation vocabulary contains only words that appear in validation-split
rows. It gets its own independent 0-indexed `row` column because fine-tuned
model embeddings are generated only for validation words — storing them in a
compact array avoids wasting space on train/test words that are not needed
for hypothesis testing.

In [ ]:
# ============================================================
# Build validation vocabulary
# ============================================================
val_def_words = set(df_val["definition_wn"].unique())
val_ans_words = set(df_val["answer_wn"].unique())
val_words = sorted(val_def_words | val_ans_words)

vocab_val = pd.DataFrame({"word": val_words, "row": range(len(val_words))})

assert vocab_val["word"].nunique() == len(vocab_val), "Duplicate words in validation vocabulary"
assert set(vocab_val["word"]).issubset(set(vocab["word"])), \
    "Validation vocabulary contains words not in full vocabulary"

vocab_val_path = DATA_DIR / "vocabulary_val.csv"
vocab_val.to_csv(vocab_val_path, index=False)

print(f"Validation vocabulary size: {len(vocab_val):,}")
print(f"  Fraction of full vocabulary: {len(vocab_val)/len(vocab):.1%}")
print(f"\nSaved to {vocab_val_path.relative_to(COMPONENT_ROOT)}")

# --- Show a few validation vocabulary entries ---
print("\nExample validation vocabulary entries:")
for _, row in vocab_val.sample(5, random_state=42).iterrows():
    print(f'  row={row["row"]:>5}, word="{row["word"]}"')


---

## §11 — Summary Statistics and Results File

Collect all coverage, split, and vocabulary statistics and write them to both
the notebook output and a standalone results file that the Architect reads to
review this notebook's outputs.

In [ ]:
# ============================================================
# Summary statistics and results file
# ============================================================
import sklearn

elapsed_total = time.time() - t0

n_input = n_before
n_output = len(df_out)
frac_retained = n_output / n_input

n_pairs_total = len(split_map)
n_pairs = {s: (split_map["split"] == s).sum() for s in ["train", "validate", "test"]}
n_rows = {s: (df_out["split"] == s).sum() for s in ["train", "validate", "test"]}

lines = [
    "# Results: 01 — WordNet Filtering and Split Assignment\n",
    "",
    "## Versions\n",
    f"- pandas: {pd.__version__}",
    f"- numpy: {np.__version__}",
    f"- scikit-learn: {sklearn.__version__}",
    f"- nltk: {nltk.__version__}",
    f"- WordNet corpus: {wn.get_version()}",
    "",
    "## Coverage\n",
    f"- Input rows (`clues_filtered.csv`): {n_input:,}",
    f"- Output rows (`clues_wn_filtered.csv`): {n_output:,}",
    f"- Fraction retained: {frac_retained:.1%}",
    f"- Total rows dropped: {n_input - n_output:,}",
    "",
    "### Article-stripping recovery (unique definitions)\n",
    f"- No strip needed: {def_diag['no_strip']:,} ({def_diag['no_strip']/def_diag['total']:.1%})",
    f'- Recovered by "a": {def_diag["a"]:,} ({def_diag["a"]/def_diag["total"]:.1%})',
    f'- Recovered by "an": {def_diag["an"]:,} ({def_diag["an"]/def_diag["total"]:.1%})',
    f'- Recovered by "the": {def_diag["the"]:,} ({def_diag["the"]/def_diag["total"]:.1%})',
    f'- Recovered by "to": {def_diag["to"]:,} ({def_diag["to"]/def_diag["total"]:.1%})',
    f"- No synsets found: {def_diag['failed']:,} ({def_diag['failed']/def_diag['total']:.1%})",
    "",
    "### Article-stripping recovery (unique answers)\n",
    f"- No strip needed: {ans_diag['no_strip']:,} ({ans_diag['no_strip']/ans_diag['total']:.1%})",
    f'- Recovered by "a": {ans_diag["a"]:,} ({ans_diag["a"]/ans_diag["total"]:.1%})',
    f'- Recovered by "an": {ans_diag["an"]:,} ({ans_diag["an"]/ans_diag["total"]:.1%})',
    f'- Recovered by "the": {ans_diag["the"]:,} ({ans_diag["the"]/ans_diag["total"]:.1%})',
    f'- Recovered by "to": {ans_diag["to"]:,} ({ans_diag["to"]/ans_diag["total"]:.1%})',
    f"- No synsets found: {ans_diag['failed']:,} ({ans_diag['failed']/ans_diag['total']:.1%})",
    "",
    "## Split Statistics\n",
    f"### Unique (definition, answer) pairs: {n_pairs_total:,}\n",
    f"- Train: {n_pairs['train']:,} ({n_pairs['train']/n_pairs_total:.1%})",
    f"- Validate: {n_pairs['validate']:,} ({n_pairs['validate']/n_pairs_total:.1%})",
    f"- Test: {n_pairs['test']:,} ({n_pairs['test']/n_pairs_total:.1%})",
    "",
    f"### Rows per split\n",
    f"- Train: {n_rows['train']:,} ({n_rows['train']/n_output:.1%})",
    f"- Validate: {n_rows['validate']:,} ({n_rows['validate']/n_output:.1%})",
    f"- Test: {n_rows['test']:,} ({n_rows['test']/n_output:.1%})",
    "",
    "## Vocabulary Statistics\n",
    f"- `vocabulary.csv`: {len(vocab):,} words",
    f"- `vocabulary_val.csv`: {len(vocab_val):,} words",
    f"- Validation vocabulary as fraction of full: {len(vocab_val)/len(vocab):.1%}",
    "",
    "### Overlap\n",
    f"- Words appearing as both definition and answer: {len(both):,}",
    f"- Words appearing only as definition: {len(only_def):,}",
    f"- Words appearing only as answer: {len(only_ans):,}",
    "",
    "## Runtime\n",
    f"- WordNet lookups: {elapsed_wn:.1f}s",
    f"- Total notebook: {elapsed_total:.1f}s",
]

results_text = "\n".join(lines) + "\n"
results_path = OUTPUT_DIR / "01_wn_filtering_and_split-results.md"
results_path.write_text(results_text)

print(results_text)
print(f"\nResults written to {results_path.relative_to(COMPONENT_ROOT)}")

---

## Summary

This notebook filtered `clues_filtered.csv` (457,262 rows) to rows where
both the definition and answer have at least one WordNet synset, producing
the foundational dataset for the custom embedding model pipeline. Roughly
52% of input rows survive — the loss is expected, driven primarily by
answers that are proper nouns, abbreviations, or informal terms absent from
WordNet.

**Article-stripping recovery:** Expanding prefix stripping from just `"a "`
(Milestone II) to include `"an "`, `"the "`, and `"to "` recovered 1,609
additional unique definitions that would otherwise have been lost.

**Split assignment:** The 30/20/50 train/validate/test split was assigned at
the (definition, answer) pair level with `random_state=42`, ensuring no
pair-level leakage across splits. Downstream f-specific datasets inherit
these assignments by subsetting.

**Output files (all in `data/filtered_split/wn_synset/`):**
- `clues_wn_filtered.csv` — filtered rows with `definition_wn`, `answer_wn`, `split`
- `clues_val.csv` — validation-split convenience subset
- `vocabulary.csv` — full unified vocabulary with canonical row ordering
- `vocabulary_val.csv` — validation-split vocabulary with own row ordering

**Results file:** `outputs/01_wn_filtering_and_split-results.md`

See the results file for exact row counts, split fractions, vocabulary
statistics, and article-stripping recovery numbers.